In [1]:
!pip install -q gradio

In [2]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


Imports and Project Paths

In [3]:
import os
import sys
import json
import re

import torch
import gradio as gr

In [4]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

TOKENIZER_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "bpe_tokenizer",
    "tokenizer.json"
)

BEST_CHECKPOINT_PATH = os.path.join(
    PROJECT_PATH,
    "models",
    "best_model.pt"
)

EVALUATION_REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs",
    "evaluation_report.json"
)

if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project path:",
    PROJECT_PATH
)

Project path: /content/drive/MyDrive/Scientific-Abstract-GPT


In [5]:
required_files = {
    "best model checkpoint": (
        BEST_CHECKPOINT_PATH
    ),
    "tokenizer": (
        TOKENIZER_PATH
    ),
    "evaluation report": (
        EVALUATION_REPORT_PATH
    )
}

missing_files = []

for file_name, file_path in (
    required_files.items()
):

    exists = os.path.exists(
        file_path
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:

        missing_files.append(
            file_path
        )

if missing_files:

    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print(
    "\nAll required files are available."
)

best model checkpoint: FOUND
tokenizer: FOUND
evaluation report: FOUND

All required files are available.


Load GPT Components and Device

In [6]:
from src.gpt_components import (
    GPTConfig,
    GPTLanguageModel,
    load_tokenizer,
    create_prompt,
    extract_abstract,
    set_seed
)

SEED = 42

set_seed(
    SEED
)

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Selected device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Selected device: cuda:0
GPU: Tesla T4


Load Tokenizer, Model and Evaluation Results

In [7]:
tokenizer = load_tokenizer(
    TOKENIZER_PATH
)

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_config = GPTConfig.from_dict(
    checkpoint[
        "model_config"
    ]
)

if (
    tokenizer.get_vocab_size()
    != model_config.vocab_size
):

    raise ValueError(
        "Tokenizer vocabulary size does not "
        "match model vocabulary size."
    )

model = GPTLanguageModel(
    model_config
).to(device)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

with open(
    EVALUATION_REPORT_PATH,
    "r",
    encoding="utf-8"
) as file:

    evaluation_report = json.load(
        file
    )

end_token_id = tokenizer.token_to_id(
    "<END>"
)

if end_token_id is None:

    raise ValueError(
        "<END> token was not found."
    )

print(
    "Model loaded successfully."
)

print(
    "Checkpoint step:",
    checkpoint["step"]
)

print(
    "Model device:",
    next(model.parameters()).device
)

print(
    "Test perplexity:",
    f"{evaluation_report['language_model_metrics']['test_perplexity']:.4f}"
)

Model loaded successfully.
Checkpoint step: 3000
Model device: cuda:0
Test perplexity: 48.4577


In [8]:
SUBJECT_OPTIONS = [
    "Artificial Intelligence",
    "Machine Learning",
    "Computation and Language"
]


def clean_generated_abstract(
    abstract
):

    abstract = abstract.replace(
        "<END>",
        ""
    )

    abstract = abstract.replace(
        "<PAD>",
        ""
    )

    abstract = abstract.replace(
        "<UNK>",
        ""
    )

    abstract = re.sub(
        r"\s+",
        " ",
        abstract
    ).strip()

    return abstract


def generate_scientific_abstract(
    title,
    subject,
    maximum_tokens,
    temperature,
    top_k,
    top_p,
    repetition_penalty
):

    title = str(
        title
    ).strip()

    subject = str(
        subject
    ).strip()

    if not title:

        return (
            "Please enter a scientific paper title.",
            "",
            ""
        )

    if subject not in SUBJECT_OPTIONS:

        return (
            "Please select a valid subject.",
            "",
            ""
        )

    maximum_tokens = int(
        maximum_tokens
    )

    top_k = int(
        top_k
    )

    prompt = create_prompt(
        title=title,
        subject=subject
    )

    encoded_prompt = tokenizer.encode(
        prompt
    )

    prompt_ids = torch.tensor(
        encoded_prompt.ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    available_tokens = (
        model_config.block_size
        - prompt_ids.shape[1]
    )

    if available_tokens <= 0:

        return (
            "The title is too long for the model context window.",
            "",
            ""
        )

    actual_maximum_tokens = min(
        maximum_tokens,
        available_tokens
    )

    with torch.inference_mode():

        generated_ids = model.generate(
            input_ids=prompt_ids,
            max_new_tokens=(
                actual_maximum_tokens
            ),
            temperature=float(
                temperature
            ),
            top_k=top_k,
            top_p=float(
                top_p
            ),
            repetition_penalty=float(
                repetition_penalty
            ),
            end_token_id=end_token_id
        )

    generated_token_list = (
        generated_ids[
            0
        ].detach().cpu().tolist()
    )

    generated_text = tokenizer.decode(
        generated_token_list,
        skip_special_tokens=False
    )

    abstract = extract_abstract(
        generated_text
    )

    abstract = clean_generated_abstract(
        abstract
    )

    generated_token_count = (
        generated_ids.shape[1]
        - prompt_ids.shape[1]
    )

    word_count = len(
        re.findall(
            r"[A-Za-z]+(?:'[A-Za-z]+)?",
            abstract
        )
    )

    status = (
        f"Generation completed successfully. "
        f"Generated tokens: {generated_token_count}. "
        f"Abstract words: {word_count}."
    )

    formatted_output = (
        f"Title: {title}\n\n"
        f"Subject: {subject}\n\n"
        f"Abstract:\n{abstract}"
    )

    return (
        abstract,
        formatted_output,
        status
    )

Test the Generation Function

In [9]:
test_abstract, test_document, test_status = (
    generate_scientific_abstract(
        title=(
            "Deep Learning for "
            "Medical Image Classification"
        ),
        subject=(
            "Machine Learning"
        ),
        maximum_tokens=180,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.1
    )
)

print(
    test_status
)

print(
    "\nGenerated abstract:\n"
)

print(
    test_abstract
)

Generation completed successfully. Generated tokens: 180. Abstract words: 157.

Generated abstract:

The potential of artificial intelligence in machine learning has led to the potential of deep learning for data mining, in terms of reliability, and privacy. However, most of their applications are often limited, which may be influenced by the data distribution. We present a new approach that can be applied to this problem as a deep neural network (DNNs) algorithm for both deep learning and the training process. Our method is based on a hybrid network called LDA, that uses this layer and then can learn the data distribution from a set of training data. Moreover, we demonstrate the efficacy of our proposed method to facilitate the development of automated deep learning models in the domain of autonomous driving. Finally, we demonstrate that the proposed approach can achieve better accuracy over the state-of-the-art deep learning methods, but also its advantages over the state-of-the-art 

Create Model Information Text

In [10]:
language_metrics = (
    evaluation_report[
        "language_model_metrics"
    ]
)

generation_metrics = (
    evaluation_report[
        "generation_metrics"
    ]
)

model_information = f"""
### Model Evaluation

- **Test loss:** {language_metrics['test_loss']:.4f}
- **Test perplexity:** {language_metrics['test_perplexity']:.4f}
- **Structural compliance:** {generation_metrics['structural_compliance_rate'] * 100:.2f}%
- **END-token completion:** {generation_metrics['end_token_completion_rate'] * 100:.2f}%
- **Average abstract length:** {generation_metrics['average_abstract_words']:.2f} words
- **Average word diversity:** {generation_metrics['average_word_diversity']:.4f}
- **Trigram repetition:** {generation_metrics['average_trigram_repetition_rate']:.4f}
"""

Build the Gradio Demo

In [11]:
with gr.Blocks(
    title=(
        "Scientific Abstract GPT"
    )
) as demo:

    gr.Markdown(
        """
        # Scientific Abstract GPT

        Generate a scientific abstract using a decoder-only
        Transformer trained on arXiv papers from Artificial
        Intelligence, Machine Learning, and Computation and Language.
        """
    )

    with gr.Row():

        with gr.Column():

            title_input = gr.Textbox(
                label="Scientific Paper Title",
                placeholder=(
                    "Enter a scientific paper title"
                ),
                value=(
                    "Deep Learning for "
                    "Medical Image Classification"
                ),
                lines=2
            )

            subject_input = gr.Dropdown(
                choices=SUBJECT_OPTIONS,
                value="Machine Learning",
                label="Research Subject"
            )

            maximum_tokens_input = gr.Slider(
                minimum=50,
                maximum=220,
                value=180,
                step=10,
                label="Maximum New Tokens"
            )

            temperature_input = gr.Slider(
                minimum=0.2,
                maximum=1.5,
                value=0.8,
                step=0.1,
                label="Temperature"
            )

            top_k_input = gr.Slider(
                minimum=0,
                maximum=100,
                value=50,
                step=5,
                label="Top-K"
            )

            top_p_input = gr.Slider(
                minimum=0.5,
                maximum=1.0,
                value=0.95,
                step=0.05,
                label="Top-P"
            )

            repetition_penalty_input = gr.Slider(
                minimum=1.0,
                maximum=1.5,
                value=1.1,
                step=0.05,
                label="Repetition Penalty"
            )

            with gr.Row():

                generate_button = gr.Button(
                    "Generate Abstract",
                    variant="primary"
                )

                clear_button = gr.ClearButton(
                    components=[
                        title_input
                    ],
                    value="Clear"
                )

        with gr.Column():

            abstract_output = gr.Textbox(
                label="Generated Abstract",
                lines=16

            )

            status_output = gr.Textbox(
                label="Generation Status",
                interactive=False
            )

    gr.Markdown(
        "## Complete Structured Output"
    )

    document_output = gr.Textbox(
        label="Generated Scientific Document",
        lines=12
    )

    gr.Markdown(
        model_information
    )

    gr.Examples(
        examples=[
            [
                (
                    "Deep Learning for "
                    "Medical Image Classification"
                ),
                "Machine Learning"
            ],
            [
                (
                    "Transformer Models for "
                    "Scientific Document Summarization"
                ),
                "Computation and Language"
            ],
            [
                (
                    "Reinforcement Learning for "
                    "Autonomous Robot Navigation"
                ),
                "Artificial Intelligence"
            ],
            [
                (
                    "Explainable Artificial Intelligence "
                    "for Healthcare Decision Support"
                ),
                "Artificial Intelligence"
            ],
            [
                (
                    "Self-Supervised Representation "
                    "Learning from Unlabeled Data"
                ),
                "Machine Learning"
            ]
        ],
        inputs=[
            title_input,
            subject_input
        ]
    )

    generate_button.click(
        fn=generate_scientific_abstract,
        inputs=[
            title_input,
            subject_input,
            maximum_tokens_input,
            temperature_input,
            top_k_input,
            top_p_input,
            repetition_penalty_input
        ],
        outputs=[
            abstract_output,
            document_output,
            status_output
        ]
    )

    title_input.submit(
        fn=generate_scientific_abstract,
        inputs=[
            title_input,
            subject_input,
            maximum_tokens_input,
            temperature_input,
            top_k_input,
            top_p_input,
            repetition_penalty_input
        ],
        outputs=[
            abstract_output,
            document_output,
            status_output
        ]
    )

print(
    "Gradio demo created successfully."
)

Gradio demo created successfully.


Launch the Demo

In [12]:
demo.queue().launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1676e0db2fd1453eca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
